# Lab 2: Projectile motion

[Start Here](../../Start_Here.ipynb) · Previous: [Lab 1: PINN fundamentals](../01_pinn/Lab_1_PINN_Fundamentals.ipynb) · Next: [Lab 3: Heat conduction](../03_heat_conduction/Lab_3_Heat_Conduction.ipynb)

Predict a projectile's position from its acceleration and initial conditions. Compare the learned motion inside and beyond the training interval.

The model takes time and returns two positions. It learns from acceleration, initial position and initial velocity; it does not advance a numerical trajectory one time step at a time.

Why do we need both initial position and velocity? After training, compare the 0–5 s error with the 5–8 s extrapolation error. There is no ground-contact rule: a negative height continues the mathematical trajectory below the launch point.

## Projectile equations

A particle starts at the origin with a given initial velocity. Find its position $(x(t),y(t))$ under constant vertical acceleration $g=-9.81$ m/s², neglecting air resistance.
<center><img src="images/projectile.svg" alt="Drawing" style="width:600px" /></center>


The analytical trajectory is: 
$$
\begin{align}
S_x &= V_{0x} t \\
S_y &= V_{0y} t + \frac{1}{2} g t^2 
\end{align}
$$
Differentiate once to obtain velocity:  
$$
\begin{align}
\frac{dS_x}{dt} &= V_{0x} \\
\frac{dS_y}{dt} &= V_{0y} + gt
\end{align}
$$
Differentiate once more to express the acceleration: 
$$
\begin{align}
\frac{\mathrm{d}^2 S_x}{\mathrm{d} t^2} &= 0 \\
\frac{\mathrm{d}^2 S_y}{\mathrm{d} t^2} &= g
\end{align}
$$
These two second-order ODEs, together with the initial position and velocity, define the training problem.

### Step 1: Time domain and initial conditions

Use $t\in[0,5]$, $v_0=40$ m/s, $\theta=\pi/3$, $x(0)=y(0)=0$, $x_t(0)=20$, and $y_t(0)=40\sin(\pi/3)$. Sample time directly; the network has no spatial input. Evaluate $5<t\le8$ separately because it is outside the training interval.

### Step 2: Symbolic equations and model

```python
class ProjectileEquation(PDE):
    def __init__(self, gravity=9.81):
        self.dim = 1
        t = Symbol("t")
        x, y = Function("x")(t), Function("y")(t)
        self.equations = {"ode_x": x.diff(t, 2), "ode_y": y.diff(t, 2) + gravity}
```

```python
class ProjectileModel(torch.nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.network = mlp(1, 2, cfg)

    def forward(self, t):
        return 100.0 * self.network(2.0 * t / 5.0 - 1.0)
```

The residual `ode_y = y_tt + 9.81` vanishes when the vertical acceleration is $-9.81$. Here `PhysicsInformer` handles spatial derivatives; the program computes time derivatives with `torch.autograd.grad` and passes them as `x__t__t` and `y__t__t`.

### Step 3: Initial and ODE losses

```python
def loss_terms(model, physics, batch_size, device):
    t = (5.0 * torch.rand(batch_size, 1, device=device, dtype=torch.float32)).requires_grad_()
    res = residuals(model(t), t, physics)
    t0 = torch.zeros(batch_size, 1, device=device, dtype=torch.float32, requires_grad=True)
    xy0 = model(t0)
    vx, vy = derivative(xy0[:, :1], t0), derivative(xy0[:, 1:], t0)
    return {"physics": sum((v / 9.81).square().mean() for v in res.values()),
            "initial_position": (xy0 / 100).square().mean(),
            "initial_velocity": ((vx - 20) / 40).square().mean()
                                + ((vy - 40 * math.sin(math.pi / 3)) / 40).square().mean()}
```

The loss has separate terms for acceleration, initial position, and initial velocity. The input-time and output scales help balance training; the plotted positions remain in metres and time in seconds.

### Step 4: Check the prediction

The analytical solution $x=20t$, $y=40\sin(\pi/3)t-9.81t^2/2$ is used only for evaluation. `heldout_before/after` checks 401 fixed times including the endpoints, along with the initial position and velocity. `in_domain_rmse` and `extrapolation_rmse` are saved separately.

The lesson accuracy checks require position RMSE at most 0.1 m, maximum position error at most 0.5 m, acceleration residual RMSE at most 0.05 m/s², initial position error at most 0.1 m, and initial velocity error at most 0.05 m/s. These limits cover the 0–5 s training interval; 5–8 s extrapolation remains a separate diagnostic and can be inaccurate even when the lesson checks pass.

### Step 5: Configuration

[config.yaml](source_code/conf/config.yaml) defines steps, batch_size, learning_rate, layer_size, and num_layers. The notebook defaults to the full 5000-step FP32 lesson run. Set `AI4SCI_STEPS=200` for an execution check; a short run is not an accuracy check. Adam's learning rate follows a cosine schedule from 0.001 toward 0.000001 so the initial conditions settle as training finishes.

```yaml
steps: 5000
batch_size: 128
learning_rate: 0.001
layer_size: 64
num_layers: 3
```

### Step 6: Train the model

Run [projectile.py](source_code/projectile.py), which uses the equations in [projectile_eqn.py](source_code/projectile_eqn.py). Its `optimize_projectile` loop clears the gradients, evaluates the loss, backpropagates, updates the weights, and decreases the learning rate after each update. The model and its derivatives use FP32 on CPU or CUDA. The saved `training_recipe` records the optimizer update count and learning rates; `accuracy` reports whether the learned trajectory meets the lesson limits.

In [ ]:
import os
import sys
import subprocess
import uuid
from pathlib import Path
import numpy as np
from IPython import get_ipython
get_ipython().run_line_magic("matplotlib", "inline")
import matplotlib.pyplot as plt
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "Start_Here.ipynb").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook inside the bootcamp repository.")
sys.path.insert(0, str(ROOT))
from ETC.runtime.notebook import show_results, validate_settings, completed_output
LAB = ROOT / "01_labs/02_projectile"
OUTPUT_BASE = Path(os.environ.get("AI4SCI_OUTPUT_DIR", str(LAB / "outputs"))).expanduser().resolve()
DEVICE = os.environ.get("AI4SCI_DEVICE", "cpu")
STEPS = int(os.environ.get("AI4SCI_STEPS", "5000"))  # full FP32 lesson run; 200 only checks execution
RUN_DIRS = {}
RUN_COMPLETED = {}
validate_settings(DEVICE, STEPS)
print("PhysicsNeMo target: 2.2.2", "device:", DEVICE, "steps:", STEPS)

In [ ]:
RUN_COMPLETED["projectile"] = False
OUTPUT = OUTPUT_BASE / ("projectile-" + uuid.uuid4().hex[:8])
RUN_DIRS["projectile"] = OUTPUT
command = [sys.executable, str(LAB / "source_code/projectile.py"), "--device", DEVICE,
           "--steps", str(STEPS), "--seed", "42", "--output-dir", str(OUTPUT)]
validate_settings(DEVICE, STEPS)
subprocess.run(command, check=True, cwd=ROOT)
RUN_COMPLETED["projectile"] = True
metrics = show_results(OUTPUT, steps=STEPS, seed=42, preview=False)

### Plot the trajectory

In [ ]:
OUTPUT = completed_output(RUN_DIRS, RUN_COMPLETED, "projectile")
data = np.load(OUTPUT / "predictions.npz", allow_pickle=False)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for j, name in enumerate(("x", "y")):
    axes[j].plot(data["t"], data["reference"][:, j], label="analytical")
    axes[j].plot(data["t"], data["prediction"][:, j], label="PINN")
    axes[j].axvline(5, color="grey", linestyle="--")
    axes[j].set(xlabel="time (s)", ylabel=name + " (m)"); axes[j].legend()
plt.show()

## View the trajectory in ParaView

The next cell compares the physical trajectory, held-out error, and actual last training batch, then exports native VTK files. Download and extract the ZIP on your computer. In [ParaView](https://www.paraview.org/download/), open `prediction.vtp` and `reference.vtp`, click **Apply**, and view from +Z. Use different solid colors, or color the prediction by `position_error_m`.

Open `validation.vtp` in **Spreadsheet View** for the held-out values in 0–5 s. Open `initial_points.vtp` and `interior_points.vtp` in a separate view: their horizontal coordinate is **time**, not the projectile's x position. The ZIP contains a short viewing guide. Values beyond 5 s are extrapolation.

In [ ]:
OUTPUT = completed_output(RUN_DIRS, RUN_COMPLETED, "projectile")
from ETC.runtime.lab_visualization import plot_projectile_training, export_projectile
from IPython.display import FileLink, display
plot_projectile_training(OUTPUT)
plt.show()
archive = export_projectile(OUTPUT)
display(FileLink(os.path.relpath(archive, Path.cwd()), result_html_prefix="Download ParaView ZIP: "))

## Inspect training with TensorBoard

Each run writes live scalar events under `outputs/_tensorboard/`. Compare `training/physics`, `training/initial_position`, and `training/initial_velocity`; a small total loss alone does not show which condition is still wrong. The curves contain actual pre-update batch losses. Use `metrics.json` for the fixed held-out accuracy check.

Set `OPEN_TENSORBOARD=True` below to open the dashboard through your authenticated Jupyter connection. No extra public port or separate login is needed. If the proxy page returns 404, the instructor must update the Launchable environment; do not expose port 6006 publicly.

In [ ]:
OPEN_TENSORBOARD = False
if OPEN_TENSORBOARD:
    from ETC.runtime.lab_visualization import tensorboard_link
    display(tensorboard_link(OUTPUT_BASE / "_tensorboard"))

### Next steps

[Start Here](../../Start_Here.ipynb) · Previous: [Lab 1: PINN fundamentals](../01_pinn/Lab_1_PINN_Fundamentals.ipynb) · Next: [Lab 3: Heat conduction](../03_heat_conduction/Lab_3_Heat_Conduction.ipynb)

--- 

Further reading: [Open Hackathons Resources](https://www.openhackathons.org/s/technical-resources). Community support: [OpenACC and Hackathons Slack Channel](https://www.openacc.org/community#slack).

---

# Licensing

Copyright © 2026 OpenACC-Standard.org. This material is released by OpenACC-Standard.org, in collaboration with NVIDIA Corporation, under the Creative Commons Attribution 4.0 International (CC BY 4.0). These materials may include references to hardware and software developed by other entities; all applicable licensing and copyrights apply.